In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 08 · Registro del modelo

> Especificación: `especificaciones/07-validacion-registro-app.md`, sección 6 · Guía de la clase 4, sección 5.

En la etapa 6 el **seguimiento** (tracking) guardó *ejecuciones*: los 60 ensayos de la búsqueda, con sus parámetros y
métricas. Eso responde "¿qué probamos?". El **registro de modelos** responde otra pregunta: **"¿cuál es el modelo que usamos,
y en qué versión?"**

| | Seguimiento (tracking) | Registro |
|---|---|---|
| Guarda | Ejecuciones (muchas, de prueba) | Versiones del modelo (pocas, candidatas a producción) |
| Pregunta | ¿Qué probamos y qué dio? | ¿Qué modelo está en uso y qué versión? |
| En TarifaLatam | 60 ensayos de `tarifalatam-optimizacion` | `tarifalatam_precio_noche`, alias `champion` |

Un **alias** (`champion`) es una etiqueta móvil: quien use el modelo pide "el champion" y no le importa qué número de
versión es. Si mañana hay uno mejor, se mueve el alias y nadie toca el código.

**Dónde:** MLflow **local** (`mlflow_local.db`), por decisión del grupo del 2026-11-05: no hay `.env` con Databricks y el
seguimiento de la etapa 6 ya está ahí. Es una excepción a la regla 7 de `CLAUDE.md`, anotada en
`especificacion-proyecto.md`. Este notebook **no lee la prueba**: usa las métricas que guardó el notebook 07.

## 0. Conexión con el seguimiento

In [2]:
from tarifalatam.config import FINAL_EXPERIMENT, MODEL_ALIAS, REGISTERED_MODEL, ROOT
from tarifalatam.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment

donde = configure_mlflow(force_local=True)  # decisión del grupo: registro local
experimento = set_experiment(FINAL_EXPERIMENT)
print("MLflow:", donde, "| experimento:", experimento)

MLflow: local | experimento: tarifalatam-modelo-final


## 1. El modelo final y los resultados de la prueba

Salen del notebook 07: el `Pipeline` completo en `models/final_pipeline.joblib` y las métricas en
`reports/07_metricas_test.json`. Las columnas de entrada son las 13 de la referencia, **sin** `host_id` ni `id`.

In [3]:
import json

import joblib
import numpy as np

from tarifalatam.config import REFERENCE_BINARY, REFERENCE_CATEGORICAL, REFERENCE_NUMERIC, REPORTS_DIR
from tarifalatam.data.load import load_train

modelo = joblib.load(ROOT / "models" / "final_pipeline.joblib")
info = json.loads((REPORTS_DIR / "07_metricas_test.json").read_text(encoding="utf-8"))
COLUMNAS_ENTRADA = REFERENCE_CATEGORICAL + REFERENCE_NUMERIC + REFERENCE_BINARY
train = load_train()
print(f"modelo: {info['decisiones']['modelo_final']} · {len(COLUMNAS_ENTRADA)} columnas de entrada")
print(f"MAE en la prueba: {info['prueba']['modelo']['mae']:.2f} USD · meta cumplida: {info['prueba']['meta']['cumple']}")

modelo: lightgbm · 13 columnas de entrada
MAE en la prueba: 41.40 USD · meta cumplida: True


## 2. Qué se registra

El modelo **con lo que hace falta para confiar en él**: parámetros, métricas de la prueba (la evaluación que se hizo una
sola vez), una **firma** (qué columnas espera y de qué tipo), un **ejemplo de entrada** y **etiquetas** de origen.

> **Nota técnica.** MLflow guarda por defecto los modelos de scikit-learn en un formato que **rechaza** un `Pipeline` con
> preprocesamiento propio. Por eso se usa `serialization_format="cloudpickle"`. Un archivo *pickle* **puede ejecutar
> código al cargarse**: carga solo modelos que guardó el grupo.

In [4]:
import mlflow
from mlflow.models import infer_signature

nombre_completo = registered_model_name(REGISTERED_MODEL)
ejemplo = train[COLUMNAS_ENTRADA].dropna().head(5)
firma = infer_signature(ejemplo, modelo.predict(ejemplo))
prueba = info["prueba"]

parametros = {"modelo": info["decisiones"]["modelo_final"], "origen_parametros": info["decisiones"]["origen_parametros"],
              "factor_log": prueba["factor_aplicado"], "columnas": len(COLUMNAS_ENTRADA)}
parametros.update({f"lgbm_{k}": v for k, v in {**info["decisiones"]["parametros"], **info["decisiones"]["parametros_fijos"]}.items()})
metricas = {f"prueba_{k}": float(v) for k, v in prueba["modelo"].items() if not isinstance(v, list)}
metricas.update({
    "prueba_mae_ic95_bajo": prueba["modelo"]["mae_ic95"][0], "prueba_mae_ic95_alto": prueba["modelo"]["mae_ic95"][1],
    "prueba_mae_linea_base": prueba["linea_base"]["mae"], "prueba_diferencia_mae": prueba["diferencia_mae"]["media"],
    "prueba_mejora_pct": prueba["meta"]["mejora_pct"], "prueba_mae_sin_filtro": prueba["sensibilidad_sin_filtro"]["mae_modelo"],
})
etiquetas = {
    "etapa": "08_registro", "datos": "Inside Airbnb, septiembre de 2026",
    "ciudades": "rio-de-janeiro, sao-paulo, mexico-city, bogota, santiago, belize",
    "objetivo": "log_price_usd (el precio en USD es exp de la predicción, sin corrección)",
    "excepcion": "T047: LightGBM elegido por el grupo después de ver la etapa 6",
    "prueba_abierta": prueba["marca"], "meta_cumplida": str(prueba["meta"]["cumple"]),
}

with mlflow.start_run(run_name="modelo-final") as run:
    mlflow.log_params(parametros)
    mlflow.log_metrics(metricas)
    mlflow.set_tags(etiquetas)
    mlflow.sklearn.log_model(
        modelo, name="model", signature=firma, input_example=ejemplo,
        registered_model_name=nombre_completo, serialization_format="cloudpickle",
    )
    run_id = run.info.run_id
print("registrado:", nombre_completo, "| ejecución:", run_id)

2026/11/05 18:12:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/11/05 18:12:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


registrado: tarifalatam_precio_noche | ejecución: 7b3281b37c0b4feabc985324d0797082


Successfully registered model 'tarifalatam_precio_noche'.
Created version '1' of model 'tarifalatam_precio_noche'.


## 3. Poner el alias `champion`

In [5]:
cliente = mlflow.MlflowClient()
version = max(int(v.version) for v in cliente.search_model_versions(f"name = '{nombre_completo}'"))
cliente.set_registered_model_alias(nombre_completo, MODEL_ALIAS, str(version))
print(f"{nombre_completo}: versión {version} -> alias '{MODEL_ALIAS}'")

tarifalatam_precio_noche: versión 1 -> alias 'champion'


## 4. Comprobación de ida y vuelta

Un registro que no se puede **recuperar** no sirve. Se carga el modelo por su alias y se comprueba que predice **igual**
que el original en 200 anuncios del entrenamiento.

In [6]:
recuperado = mlflow.sklearn.load_model(f"models:/{nombre_completo}@{MODEL_ALIAS}")
muestra = train[COLUMNAS_ENTRADA].head(200)
iguales = np.isclose(modelo.predict(muestra), recuperado.predict(muestra))
assert iguales.all(), "el modelo recuperado del registro no coincide con el original"
print(f"el modelo recuperado del registro predice igual que el original en {int(iguales.sum())} de {len(muestra)} anuncios")

el modelo recuperado del registro predice igual que el original en 200 de 200 anuncios


## 5. Guardar la constancia

In [7]:
import time

constancia = {
    "tracking": donde, "experimento": experimento, "modelo_registrado": nombre_completo, "version": version,
    "alias": MODEL_ALIAS, "run_id": run_id, "registrado_el": time.strftime("%Y-%m-%d %H:%M:%S"),
    "artefactos": f"mlruns/{experimento} (en .gitignore; la ruta absoluta queda en mlflow_local.db)",
    "ida_y_vuelta": {"filas": int(len(muestra)), "iguales": int(iguales.sum())},
    "como_recuperarlo": f"configure_mlflow(force_local=True); mlflow.sklearn.load_model('models:/{nombre_completo}@{MODEL_ALIAS}')",
}
(REPORTS_DIR / "08_registro.json").write_text(json.dumps(constancia, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(constancia, indent=2, ensure_ascii=False))

{
  "tracking": "local",
  "experimento": "tarifalatam-modelo-final",
  "modelo_registrado": "tarifalatam_precio_noche",
  "version": 1,
  "alias": "champion",
  "run_id": "7b3281b37c0b4feabc985324d0797082",
  "registrado_el": "2026-11-05 18:12:34",
  "artefactos": "mlruns/tarifalatam-modelo-final (en .gitignore; la ruta absoluta queda en mlflow_local.db)",
  "ida_y_vuelta": {
    "filas": 200,
    "iguales": 200
  },
  "como_recuperarlo": "configure_mlflow(force_local=True); mlflow.sklearn.load_model('models:/tarifalatam_precio_noche@champion')"
}


## Qué sigue

El notebook 09 exporta el modelo `champion` a `models/model.joblib`, que viaja con la app. **La app no consulta MLflow ni
Databricks**: el registro sigue siendo la **fuente de verdad** y el archivo siempre se exporta desde la versión `champion`.